# ResNet-50 CAM 실험
담당: 이희선


## 셀 1. 데이터 다운로드 

In [ ]:
import os

DATA_DIR = os.path.expanduser("~/work/class_activation_map")
IMG_DIR  = os.path.join(DATA_DIR, "Images")

if not os.path.exists(IMG_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar     -P {DATA_DIR}
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/annotation.tar -P {DATA_DIR}
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/lists.tar      -P {DATA_DIR}
    !tar -xf {DATA_DIR}/images.tar     -C {DATA_DIR}
    !tar -xf {DATA_DIR}/annotation.tar -C {DATA_DIR}
    !tar -xf {DATA_DIR}/lists.tar      -C {DATA_DIR}
    print("데이터 다운로드 완료!")
else:
    print("이미 데이터가 존재합니다. 다운로드 건너뜁니다.")

!ls {DATA_DIR}

In [ ]:
import os

DATA_DIR = os.path.expanduser("~/work/class_activation_map")
IMG_DIR  = os.path.join(DATA_DIR, "Images")

if not os.path.exists(IMG_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar     -P {DATA_DIR}
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/annotation.tar -P {DATA_DIR}
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/lists.tar      -P {DATA_DIR}
    !tar -xf {DATA_DIR}/images.tar     -C {DATA_DIR}
    !tar -xf {DATA_DIR}/annotation.tar -C {DATA_DIR}
    !tar -xf {DATA_DIR}/lists.tar      -C {DATA_DIR}
    print("데이터 다운로드 완료!")
else:
    print("이미 데이터가 존재합니다. 다운로드 건너뜁니다.")

!ls {DATA_DIR}

In [ ]:
import os

DATA_DIR = os.path.expanduser("~/work/class_activation_map")
IMG_DIR  = os.path.join(DATA_DIR, "Images")

if not os.path.exists(IMG_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar     -P {DATA_DIR}
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/annotation.tar -P {DATA_DIR}
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/lists.tar      -P {DATA_DIR}
    !tar -xf {DATA_DIR}/images.tar     -C {DATA_DIR}
    !tar -xf {DATA_DIR}/annotation.tar -C {DATA_DIR}
    !tar -xf {DATA_DIR}/lists.tar      -C {DATA_DIR}
    print("데이터 다운로드 완료!")
else:
    print("이미 데이터가 존재합니다. 다운로드 건너뜁니다.")

!ls {DATA_DIR}

In [ ]:
import os

DATA_DIR = os.path.expanduser("~/work/class_activation_map")
IMG_DIR  = os.path.join(DATA_DIR, "Images")

if not os.path.exists(IMG_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar     -P {DATA_DIR}
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/annotation.tar -P {DATA_DIR}
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/lists.tar      -P {DATA_DIR}
    !tar -xf {DATA_DIR}/images.tar     -C {DATA_DIR}
    !tar -xf {DATA_DIR}/annotation.tar -C {DATA_DIR}
    !tar -xf {DATA_DIR}/lists.tar      -C {DATA_DIR}
    print("데이터 다운로드 완료!")
else:
    print("이미 데이터가 존재합니다. 다운로드 건너뜁니다.")

!ls {DATA_DIR}

In [ ]:
import os

DATA_DIR = os.path.expanduser("~/work/class_activation_map")
IMG_DIR  = os.path.join(DATA_DIR, "Images")

if not os.path.exists(IMG_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar     -P {DATA_DIR}
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/annotation.tar -P {DATA_DIR}
    !wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/lists.tar      -P {DATA_DIR}
    !tar -xf {DATA_DIR}/images.tar     -C {DATA_DIR}
    !tar -xf {DATA_DIR}/annotation.tar -C {DATA_DIR}
    !tar -xf {DATA_DIR}/lists.tar      -C {DATA_DIR}
    print("데이터 다운로드 완료!")
else:
    print("이미 데이터가 존재합니다. 다운로드 건너뜁니다.")

!ls {DATA_DIR}

In [ ]:
!pip install opencv-python-headless

## 셀 2. 라이브러리 임포트 & 버전 확인

In [ ]:
import os, json, random, glob
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import scipy.io

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

print(f"torch      : {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"numpy      : {np.__version__}")
print(f"cv2        : {cv2.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device     : {device}")

## 셀 3. 공통 Config 

In [ ]:
# ── 두 팀이 반드시 동일한 값을 사용해야 비교가 의미 있습니다 ──────────────────
SEED              = 42
INPUT_SIZE        = 224
IMAGENET_MEAN     = [0.485, 0.456, 0.406]
IMAGENET_STD      = [0.229, 0.224, 0.225]
BATCH_SIZE        = 32
NUM_EPOCHS        = 10
LR                = 1e-3
CAM_ALPHA         = 0.5
THRESHOLD_LIST    = [0.3, 0.5, 0.7]
THRESHOLD_DEFAULT = 0.5
SAVE_DIR          = "./outputs"
DATA_DIR          = os.path.expanduser("~/work/class_activation_map")

# 평가 샘플 인덱스 — 셀 6 preview 후 팀과 함께 확정
EVAL_INDICES = {
    "easy": [4, 11, 23, 38, 55, 62, 77, 89, 103, 117],
    "hard": [6, 14, 29, 41, 58, 70, 85, 96, 108, 120],
}

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
print("Config 설정 완료")

## 셀 4. 재현성 고정
같은 SEED 를 써야 두 팀의 DataLoader 셔플 순서가 동일해집니다.

In [ ]:
def fix_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

fix_seed()
print(f"Seed {SEED} 고정 완료")

## 셀 5. Stanford Dogs 커스텀 Dataset

`torchvision.datasets.StanfordDogs` 는 bbox 를 제공하지 않으므로  
annotation XML 을 직접 파싱하는 Dataset 을 만듭니다.

**폴더 구조**
```
~/work/class_activation_map/
  Images/        ← 견종 폴더별 jpg
  Annotation/    ← 견종 폴더별 xml (bbox 포함)
  train_list.mat
  test_list.mat
```

In [ ]:
def parse_bbox(annotation_path):
    tree   = ET.parse(annotation_path)
    root   = tree.getroot()
    bndbox = root.find(".//bndbox")
    return (
        int(bndbox.find("xmin").text),
        int(bndbox.find("ymin").text),
        int(bndbox.find("xmax").text),
        int(bndbox.find("ymax").text),
    )


class StanfordDogsDataset(Dataset):
    def __init__(self, data_dir, split="train", transform=None):
        self.data_dir  = data_dir
        self.transform = transform

        mat_path = os.path.join(data_dir,
                    "train_list.mat" if split == "train" else "test_list.mat")
        mat = scipy.io.loadmat(mat_path)

        self.file_list = [str(mat["file_list"][i][0][0])
                          for i in range(len(mat["file_list"]))]
        self.labels    = [int(mat["labels"][i][0]) - 1
                          for i in range(len(mat["labels"]))]
        self.classes   = sorted(set(p.split("/")[0] for p in self.file_list))

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        rel_path = self.file_list[idx]
        label    = self.labels[idx]

        img_path = os.path.join(self.data_dir, "Images", rel_path)
        if not os.path.exists(img_path):
            img_path = img_path + ".jpg"

        pil_orig = Image.open(img_path).convert("RGB")
        orig_w, orig_h = pil_orig.size

        ann_rel  = rel_path.replace(".jpg", "")
        ann_path = os.path.join(self.data_dir, "Annotation", ann_rel)
        raw_bbox = parse_bbox(ann_path)

        sx = INPUT_SIZE / orig_w
        sy = INPUT_SIZE / orig_h
        gt_bbox = (
            int(raw_bbox[0] * sx), int(raw_bbox[1] * sy),
            int(raw_bbox[2] * sx), int(raw_bbox[3] * sy),
        )

        pil_224 = pil_orig.resize((INPUT_SIZE, INPUT_SIZE), Image.BILINEAR)

        if self.transform:
            img_tensor = self.transform(pil_224)
        else:
            img_tensor = transforms.ToTensor()(pil_224)

        return img_tensor, label, gt_bbox, pil_224

## 셀 6. DataLoader 생성

In [ ]:
# train: RandomHorizontalFlip 만 허용 (두 팀 동일)
train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = StanfordDogsDataset(DATA_DIR, split="train", transform=train_transform)
val_dataset   = StanfordDogsDataset(DATA_DIR, split="test",  transform=val_transform)

# gt_bbox 가 tuple 이라 커스텀 collate_fn 필요
def collate_fn(batch):
    imgs   = torch.stack([b[0] for b in batch])
    labels = torch.tensor([b[1] for b in batch])
    bboxes = [b[2] for b in batch]
    pils   = [b[3] for b in batch]
    return imgs, labels, bboxes, pils

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, collate_fn=collate_fn)

print(f"Train: {len(train_dataset)}장  |  Val: {len(val_dataset)}장")
print(f"클래스 수: {len(train_dataset.classes)}")

# 샘플 하나 확인
img_t, lbl, bbox, pil = val_dataset[0]
print(f"\n샘플 확인 → tensor:{img_t.shape} | label:{lbl} | gt_bbox:{bbox}")
plt.imshow(pil); plt.title(f"label={lbl}  bbox={bbox}"); plt.axis("off"); plt.show()

## 셀 7. EVAL_INDICES 샘플 미리보기

**GT bbox 가 표시된 이미지를 눈으로 확인하고, 팀과 함께 easy/hard 인덱스를 확정한 뒤  
셀 3의 `EVAL_INDICES` 를 수정하세요.**

In [ ]:
def preview_eval_samples(dataset, eval_indices=EVAL_INDICES):
    all_idx = eval_indices["easy"] + eval_indices["hard"]
    fig, axes = plt.subplots(2, 10, figsize=(22, 5))
    fig.suptitle("상단: easy  /  하단: hard  (초록 박스 = GT bbox)", fontsize=11)
    for i, idx in enumerate(all_idx):
        row, col = divmod(i, 10)
        _, lbl, bbox, pil = dataset[idx]
        draw = np.array(pil).copy()
        x1,y1,x2,y2 = bbox
        cv2.rectangle(draw, (x1,y1), (x2,y2), (0,220,0), 2)
        axes[row][col].imshow(draw)
        axes[row][col].set_title(f"idx={idx}\nlbl={lbl}", fontsize=7)
        axes[row][col].axis("off")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/preview_eval_samples.png", dpi=120)
    plt.show()
    print("→ 확인 후 셀 3의 EVAL_INDICES 를 팀과 함께 확정하세요.")

preview_eval_samples(val_dataset)

## 셀 8. 모델 정의 — ResNet-50 + GAP

**CAM 을 생성하려면 마지막 conv 뒤에 GAP → FC 구조가 필요합니다.**

| | 기존 ResNet-50 | 수정 후 |
|---|---|---|
| pooling | AvgPool(7×7) | GAP(1×1) |
| FC | 2048 → 1000 | 2048 → 120 |

In [ ]:
class ResNet50CAM(nn.Module):
    def __init__(self, num_classes=120, pretrained=True):
        super().__init__()
        base = models.resnet50(pretrained=pretrained)
        # conv1 + bn1 + relu + maxpool
        self.layer0 = nn.Sequential(*list(base.children())[:4])
        self.layer1 = base.layer1   # 출력: (B, 256,  56, 56)
        self.layer2 = base.layer2   # 출력: (B, 512,  28, 28)
        self.layer3 = base.layer3   # 출력: (B, 1024, 14, 14)
        self.layer4 = base.layer4   # 출력: (B, 2048,  7,  7)
        self.gap    = nn.AdaptiveAvgPool2d(1)   # GAP → (B, 2048, 1, 1)
        self.fc     = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gap(x).flatten(1)
        return self.fc(x)

    def get_layer(self, name):
        """레이어 이름으로 레이어 반환 (Grad-CAM target 지정용)."""
        return {"layer1": self.layer1, "layer2": self.layer2,
                "layer3": self.layer3, "layer4": self.layer4}[name]

model = ResNet50CAM(num_classes=120, pretrained=True).to(device)
print("모델 생성 완료")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

## 셀 9. Phase 0 — 학습

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels, _, _ in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels, _, _ in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out  = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history  = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc = validate(model, val_loader, criterion)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"train loss={tr_loss:.4f} acc={tr_acc:.4f} | "
          f"val loss={vl_loss:.4f} acc={vl_acc:.4f}")

    if vl_acc > best_acc:
        best_acc = vl_acc
        torch.save(model.state_dict(), f"{SAVE_DIR}/best_model_resnet50.pt")
        print(f"  → best model saved (val_acc={best_acc:.4f})")

with open(f"{SAVE_DIR}/training_history_resnet50.json", "w") as f:
    json.dump(history, f, indent=2)
print("\n학습 완료!")

## 셀 10. 러닝커브

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history["train_loss"], label="Train")
ax1.plot(epochs, history["val_loss"],   label="Val")
ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend()

ax2.plot(epochs, history["train_acc"], label="Train")
ax2.plot(epochs, history["val_acc"],   label="Val")
ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.legend()

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/learning_curve_resnet50.png", dpi=150)
plt.show()

## 셀 11. 최적 모델 로드

In [ ]:
model.load_state_dict(torch.load(f"{SAVE_DIR}/best_model_resnet50.pt",
                                  map_location=device))
model.eval()
print("best_model_resnet50.pt 로드 완료")

## 셀 12. 공통 유틸

In [ ]:
def unnormalize(tensor):
    """ImageNet 정규화 역변환 → RGB uint8 numpy (H,W,3)."""
    t    = tensor.squeeze(0).clone().cpu()
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    t    = (t * std + mean).clamp(0, 1)
    return (t.permute(1,2,0).numpy() * 255).astype(np.uint8)

def get_iou(gt_bbox, pred_bbox):
    """IoU = 교집합 넓이 / 합집합 넓이."""
    gx1,gy1,gx2,gy2 = gt_bbox
    px1,py1,px2,py2 = pred_bbox
    ix1,iy1 = max(gx1,px1), max(gy1,py1)
    ix2,iy2 = min(gx2,px2), min(gy2,py2)
    inter   = max(0, ix2-ix1) * max(0, iy2-iy1)
    union   = (gx2-gx1)*(gy2-gy1) + (px2-px1)*(py2-py1) - inter
    return float(inter/union) if union > 0 else 0.0

print("유틸 함수 정의 완료")

## 셀 13. CAM 생성

**원리:** `CAM = Σ_c ( fc_weight_c × feature_map_c )`  
FC 레이어의 가중치와 마지막 conv feature map 을 채널별로 가중합합니다.

In [ ]:
def generate_cam(model, img_tensor, class_idx=None):
    """
    Args:
        img_tensor : (1, 3, 224, 224) 정규화된 텐서
        class_idx  : None 이면 예측 클래스 사용
    Returns:
        cam_map : uint8 numpy (7×7)
    """
    model.eval()
    feat_map = {}
    # forward hook 으로 layer4 의 출력(feature map) 캡처
    hook = model.layer4.register_forward_hook(
        lambda m, i, o: feat_map.update({"v": o.detach()}))

    with torch.no_grad():
        logits = model(img_tensor)
    hook.remove()

    if class_idx is None:
        class_idx = logits.argmax(1).item()

    # FC 가중치에서 해당 클래스 행만 추출: (2048,)
    fc_w  = model.fc.weight[class_idx].detach().cpu()
    # layer4 feature map: (2048, 7, 7)
    feats = feat_map["v"].squeeze(0).cpu()

    # 채널별 가중합 → (7, 7)
    cam = torch.einsum("c,chw->hw", fc_w, feats)
    cam = F.relu(cam)                             # 음수 제거
    cam = (cam - cam.min()) / (cam.max() + 1e-8)  # [0,1] 정규화

    return (cam.numpy() * 255).astype(np.uint8)

print("generate_cam 정의 완료")

## 셀 14. Grad-CAM 생성

**원리:**  
1. 타겟 레이어에 forward/backward hook 등록  
2. 클래스 스코어에 대해 역전파  
3. gradient 를 채널별 GAP → 중요도 가중치 `w_c`  
4. `Grad-CAM = ReLU( Σ_c w_c × F_c )`

In [ ]:
def generate_grad_cam(model, layer_name, img_tensor, class_idx=None):
    """
    Args:
        layer_name : "layer1" / "layer2" / "layer3" / "layer4"
        img_tensor : (1, 3, 224, 224)
        class_idx  : None 이면 예측 클래스 사용
    Returns:
        grad_cam_map : uint8 numpy (레이어 크기에 따라 다름)
    """
    model.eval()
    target = model.get_layer(layer_name)
    acts, grads = {}, {}

    # forward hook: activation 저장
    fwd = target.register_forward_hook(
        lambda m, i, o: acts.update({"v": o.detach()}))
    # backward hook: gradient 저장
    bwd = target.register_full_backward_hook(
        lambda m, gi, go: grads.update({"v": go[0].detach()}))

    logits = model(img_tensor)
    if class_idx is None:
        class_idx = logits.argmax(1).item()

    model.zero_grad()
    logits[0, class_idx].backward()   # 타겟 클래스 스코어에 대해서만 역전파
    fwd.remove(); bwd.remove()

    feats   = acts["v"].squeeze(0).cpu()                    # (C, H, W)
    weights = grads["v"].squeeze(0).cpu().mean(dim=(1, 2))  # (C,) — 채널별 GAP

    gcam = torch.einsum("c,chw->hw", weights, feats)
    gcam = F.relu(gcam)
    gcam = (gcam - gcam.min()) / (gcam.max() + 1e-8)

    return (gcam.numpy() * 255).astype(np.uint8)

print("generate_grad_cam 정의 완료")

## 셀 15. BBox 형성 방식 3가지 (실험 3번)

| 방식 | 설명 | 장점 | 단점 |
|---|---|---|---|
| argwhere | threshold 이상 픽셀의 min/max | 단순 직관적 | 노이즈에 취약 |
| connected_component | 가장 큰 연결 성분 | 노이즈 무시 | 기본값으로 사용 |
| contour | 모든 윤곽선 합쳐 외접 사각형 | 넓은 커버 | 배경 포함 가능 |

In [ ]:
def get_bbox_argwhere(cam, orig_size, threshold=THRESHOLD_DEFAULT):
    """방법1: threshold 이상 픽셀 좌표의 min/max → bbox."""
    orig_w, orig_h = orig_size
    r = cv2.resize(cam, (orig_w, orig_h))
    coords = np.argwhere(r >= int(r.max() * threshold))
    if len(coords) == 0:
        return (0, 0, orig_w, orig_h)
    y1, x1 = coords.min(axis=0)
    y2, x2 = coords.max(axis=0)
    return (int(x1), int(y1), int(x2), int(y2))

def get_bbox_connected_component(cam, orig_size, threshold=THRESHOLD_DEFAULT):
    """방법2: 이진화 후 가장 큰 연결 성분의 외접 사각형 → bbox. (기본값)"""
    orig_w, orig_h = orig_size
    r = cv2.resize(cam, (orig_w, orig_h))
    _, binary = cv2.threshold(r, int(r.max() * threshold), 255, cv2.THRESH_BINARY)
    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if num_labels <= 1:
        return (0, 0, orig_w, orig_h)
    # 0번은 배경, 1번부터 탐색 → 가장 넓은 성분 선택
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    x = stats[largest, cv2.CC_STAT_LEFT]
    y = stats[largest, cv2.CC_STAT_TOP]
    w = stats[largest, cv2.CC_STAT_WIDTH]
    h = stats[largest, cv2.CC_STAT_HEIGHT]
    return (x, y, x+w, y+h)

def get_bbox_contour(cam, orig_size, threshold=THRESHOLD_DEFAULT):
    """방법3: 모든 윤곽선을 합쳐 최소 외접 사각형 → bbox."""
    orig_w, orig_h = orig_size
    r = cv2.resize(cam, (orig_w, orig_h))
    _, binary = cv2.threshold(r, int(r.max() * threshold), 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return (0, 0, orig_w, orig_h)
    x, y, w, h = cv2.boundingRect(np.concatenate(contours))
    return (x, y, x+w, y+h)

BBOX_METHODS = {
    "argwhere":            get_bbox_argwhere,
    "connected_component": get_bbox_connected_component,
    "contour":             get_bbox_contour,
}
print("bbox 함수 3가지 정의 완료")

## 셀 16. 시각화 유틸

In [ ]:
def _heatmap(cam, size):
    """cam → Jet 컬러맵 RGB, size=(W,H)."""
    r = cv2.resize(cam, size, interpolation=cv2.INTER_LINEAR)
    return cv2.cvtColor(cv2.applyColorMap(r, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)

def overlay_cam(cam, img_rgb, alpha=CAM_ALPHA):
    """히트맵과 원본 이미지를 alpha 비율로 블렌딩. 반환: RGB uint8."""
    h, w = img_rgb.shape[:2]
    return cv2.addWeighted(
        _heatmap(cam, (w,h)).astype(np.float32), alpha,
        img_rgb.astype(np.float32), 1-alpha, 0).astype(np.uint8)

def visualize_comparison(img_rgb, cam_map, gcam_map,
                          cam_bbox, gcam_bbox, gt_bbox,
                          sample_id="", save_path=None):
    """4장 나란히: [원본+GT] [CAM overlay] [Grad-CAM overlay] [bbox 비교]"""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f"Sample {sample_id}", fontsize=12)

    # ① 원본 + GT bbox
    axes[0].imshow(img_rgb)
    axes[0].add_patch(mpatches.Rectangle(
        (gt_bbox[0],gt_bbox[1]), gt_bbox[2]-gt_bbox[0], gt_bbox[3]-gt_bbox[1],
        lw=2, edgecolor="lime", facecolor="none"))
    axes[0].set_title("Original + GT")

    # ② CAM overlay
    axes[1].imshow(overlay_cam(cam_map, img_rgb))
    axes[1].set_title("CAM overlay")

    # ③ Grad-CAM overlay
    axes[2].imshow(overlay_cam(gcam_map, img_rgb))
    axes[2].set_title("Grad-CAM overlay (layer4)")

    # ④ bbox 비교
    axes[3].imshow(img_rgb)
    for bbox, color, lbl in [(gt_bbox,"lime","GT"),
                              (cam_bbox,"red","CAM"),
                              (gcam_bbox,"deepskyblue","Grad-CAM")]:
        axes[3].add_patch(mpatches.Rectangle(
            (bbox[0],bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1],
            lw=2, edgecolor=color, facecolor="none", label=lbl))
    axes[3].legend(loc="upper right", fontsize=8)
    axes[3].set_title("BBox comparison")

    for ax in axes: ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()

print("시각화 함수 정의 완료")

## 셀 17. 실험 1 — 레이어별 Grad-CAM

얕은 레이어(layer1): 엣지·텍스처 등 저수준 특징  
깊은 레이어(layer4): 클래스와 관련된 고수준 의미 특징  
→ **레이어가 깊어질수록 localization 이 정확해지는지 확인**

In [ ]:
def experiment1_layer(model, img_tensor, img_rgb, gt_bbox, sample_id=""):
    orig_size = (img_rgb.shape[1], img_rgb.shape[0])
    layers    = ["layer1", "layer2", "layer3", "layer4"]

    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    fig.suptitle(f"[실험1] 레이어별 Grad-CAM  |  Sample {sample_id}", fontsize=12)

    # 원본
    axes[0].imshow(img_rgb)
    axes[0].add_patch(mpatches.Rectangle(
        (gt_bbox[0],gt_bbox[1]), gt_bbox[2]-gt_bbox[0], gt_bbox[3]-gt_bbox[1],
        lw=2, edgecolor="lime", facecolor="none"))
    axes[0].set_title("Original + GT"); axes[0].axis("off")

    iou_by_layer = {}
    for ax, name in zip(axes[1:], layers):
        gcam = generate_grad_cam(model, name, img_tensor)
        bbox = get_bbox_connected_component(gcam, orig_size)
        iou  = get_iou(gt_bbox, bbox)
        iou_by_layer[name] = round(iou, 4)
        ax.imshow(overlay_cam(gcam, img_rgb))
        ax.add_patch(mpatches.Rectangle(
            (bbox[0],bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1],
            lw=2, edgecolor="red", facecolor="none"))
        ax.set_title(f"{name}\nIoU={iou:.3f}"); ax.axis("off")

    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/exp1_layer_{sample_id}.png", dpi=100, bbox_inches="tight")
    plt.show()
    print(f"  레이어별 IoU: {iou_by_layer}")
    return iou_by_layer

print("experiment1_layer 정의 완료")

## 셀 18. 실험 2 — Threshold 변경

낮은 threshold(0.3): 더 많은 픽셀 포함 → bbox 크게 → recall ↑  
높은 threshold(0.7): 핵심 픽셀만 포함 → bbox 작게 → precision ↑

In [ ]:
def experiment2_threshold(model, img_tensor, img_rgb, gt_bbox, sample_id=""):
    orig_size = (img_rgb.shape[1], img_rgb.shape[0])
    cam_map   = generate_cam(model, img_tensor)
    gcam_map  = generate_grad_cam(model, "layer4", img_tensor)

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(f"[실험2] Threshold 변경  |  Sample {sample_id}", fontsize=12)

    iou_by_thr = {"cam": {}, "gradcam": {}}
    for row, (m, lbl, key) in enumerate([
            (cam_map,  "CAM",      "cam"),
            (gcam_map, "Grad-CAM", "gradcam")]):

        axes[row][0].imshow(img_rgb)
        axes[row][0].set_title(f"{lbl} - Original"); axes[row][0].axis("off")

        for col, thr in enumerate(THRESHOLD_LIST, start=1):
            bbox = get_bbox_connected_component(m, orig_size, threshold=thr)
            iou  = get_iou(gt_bbox, bbox)
            iou_by_thr[key][thr] = round(iou, 4)
            axes[row][col].imshow(overlay_cam(m, img_rgb))
            axes[row][col].add_patch(mpatches.Rectangle(
                (bbox[0],bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1],
                lw=2, edgecolor="red", facecolor="none"))
            axes[row][col].set_title(f"thr={thr}\nIoU={iou:.3f}")
            axes[row][col].axis("off")

    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/exp2_thr_{sample_id}.png", dpi=100, bbox_inches="tight")
    plt.show()
    print(f"  threshold별 IoU: {iou_by_thr}")
    return iou_by_thr

print("experiment2_threshold 정의 완료")

## 셀 19. 실험 3 — BBox 형성 방식 비교

threshold = 0.5 고정, bbox 방식(argwhere / connected_component / contour)만 변경

In [ ]:
def experiment3_bbox_method(model, img_tensor, img_rgb, gt_bbox, sample_id=""):
    orig_size = (img_rgb.shape[1], img_rgb.shape[0])
    cam_map   = generate_cam(model, img_tensor)
    gcam_map  = generate_grad_cam(model, "layer4", img_tensor)

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(f"[실험3] BBox 방식 비교  |  Sample {sample_id}", fontsize=12)

    iou_by_method = {"cam": {}, "gradcam": {}}
    for row, (m, lbl, key) in enumerate([
            (cam_map,  "CAM",      "cam"),
            (gcam_map, "Grad-CAM", "gradcam")]):

        axes[row][0].imshow(img_rgb)
        axes[row][0].set_title(f"{lbl} - Original"); axes[row][0].axis("off")

        for col, (mname, mfn) in enumerate(BBOX_METHODS.items(), start=1):
            bbox = mfn(m, orig_size, threshold=THRESHOLD_DEFAULT)
            iou  = get_iou(gt_bbox, bbox)
            iou_by_method[key][mname] = round(iou, 4)
            axes[row][col].imshow(overlay_cam(m, img_rgb))
            axes[row][col].add_patch(mpatches.Rectangle(
                (bbox[0],bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1],
                lw=2, edgecolor="red", facecolor="none"))
            axes[row][col].set_title(f"{mname}\nIoU={iou:.3f}")
            axes[row][col].axis("off")

    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/exp3_bbox_{sample_id}.png", dpi=100, bbox_inches="tight")
    plt.show()
    print(f"  방식별 IoU: {iou_by_method}")
    return iou_by_method

print("experiment3_bbox_method 정의 완료")

## 셀 20. Phase 1 — 전체 평가 실행 (output1 + output2 자동 생성)

In [ ]:
all_indices = EVAL_INDICES["easy"] + EVAL_INDICES["hard"]
records = []   # output1 정량표용

for idx in all_indices:
    print(f"\n{'='*55}")
    diff = 'easy' if idx in EVAL_INDICES['easy'] else 'hard'
    print(f"  Sample idx={idx}  ({diff})")
    print(f"{'='*55}")

    img_t, lbl, gt_bbox, pil = val_dataset[idx]
    img_tensor = img_t.unsqueeze(0).to(device)
    img_rgb    = np.array(pil)
    orig_size  = (img_rgb.shape[1], img_rgb.shape[0])

    # ── Phase 1 기본 비교 (CAM vs Grad-CAM, layer4, threshold=0.5) ──────────
    cam_map   = generate_cam(model, img_tensor)
    gcam_map  = generate_grad_cam(model, "layer4", img_tensor)
    cam_bbox  = get_bbox_connected_component(cam_map,  orig_size)
    gcam_bbox = get_bbox_connected_component(gcam_map, orig_size)
    cam_iou   = get_iou(gt_bbox, cam_bbox)
    gcam_iou  = get_iou(gt_bbox, gcam_bbox)

    records.append({"sample_id": idx, "label": lbl,
                     "cam_iou": cam_iou, "gradcam_iou": gcam_iou})

    # output2: 정성 시각화
    visualize_comparison(img_rgb, cam_map, gcam_map,
                          cam_bbox, gcam_bbox, gt_bbox,
                          sample_id=idx,
                          save_path=f"{SAVE_DIR}/output2_sample_{idx}.png")

    # 실험 1 — 레이어별
    experiment1_layer(model, img_tensor, img_rgb, gt_bbox, sample_id=idx)
    # 실험 2 — threshold
    experiment2_threshold(model, img_tensor, img_rgb, gt_bbox, sample_id=idx)
    # 실험 3 — bbox 방식
    experiment3_bbox_method(model, img_tensor, img_rgb, gt_bbox, sample_id=idx)

print("\n전체 평가 완료!")

## 셀 21. output1 — 정량표 출력 & 저장

In [ ]:
print(f"\n{'='*55}")
print(f"  [Output 1]  ResNet-50  정량 분석표")
print(f"{'='*55}")
print(f"{'idx':>6} | {'난이도':^6} | {'CAM IoU':>10} | {'Grad-CAM IoU':>12}")
print("-" * 42)
for r in records:
    diff = "easy" if r["sample_id"] in EVAL_INDICES["easy"] else "hard"
    print(f"{r['sample_id']:>6} | {diff:^6} | "
          f"{r['cam_iou']:>10.4f} | {r['gradcam_iou']:>12.4f}")

cam_mean  = np.mean([r["cam_iou"]     for r in records])
gcam_mean = np.mean([r["gradcam_iou"] for r in records])
easy_cam  = np.mean([r["cam_iou"]     for r in records if r["sample_id"] in EVAL_INDICES["easy"]])
hard_cam  = np.mean([r["cam_iou"]     for r in records if r["sample_id"] in EVAL_INDICES["hard"]])
easy_gcam = np.mean([r["gradcam_iou"] for r in records if r["sample_id"] in EVAL_INDICES["easy"]])
hard_gcam = np.mean([r["gradcam_iou"] for r in records if r["sample_id"] in EVAL_INDICES["hard"]])

print("-" * 42)
print(f"{'easy 평균':>6} |        | {easy_cam:>10.4f} | {easy_gcam:>12.4f}")
print(f"{'hard 평균':>6} |        | {hard_cam:>10.4f} | {hard_gcam:>12.4f}")
print(f"{'전체 평균':>6} |        | {cam_mean:>10.4f} | {gcam_mean:>12.4f}")
print(f"{'='*55}")

# JSON 저장 (팀 공유용)
with open(f"{SAVE_DIR}/iou_results_resnet50.json", "w") as f:
    json.dump({"records": records,
               "cam_mean": cam_mean,   "gradcam_mean": gcam_mean,
               "easy_cam": easy_cam,   "hard_cam": hard_cam,
               "easy_gcam": easy_gcam, "hard_gcam": hard_gcam}, f, indent=2)
print(f"\n결과 저장 완료 → {SAVE_DIR}/iou_results_resnet50.json")

## 회고 

### 학습 결과

10 epoch 동안 학습했는데 val_acc가 0.26 정도 나왔다. 120개 클래스 중에서 맞추는 거라 처음엔 낮아 보였는데, 사실 bbox 라벨 없이 분류만 학습한 모델치고는 나쁘지 않은 수준이라고 한다. 러닝커브를 보면 epoch 10에서도 아직 올라가는 추세여서 epoch를 더 늘리면 성능이 더 좋아질 것 같다.

---

### 실험 결과 (IoU 기준)

|  | CAM | Grad-CAM |
|---|---|---|
| easy 평균 | 0.4687 | 0.4682 |
| hard 평균 | 0.3467 | 0.3467 |
| 전체 평균 | 0.4077 | 0.4075 |

**1. CAM이랑 Grad-CAM이 거의 똑같다**

대부분의 샘플에서 소수점 4자리까지 동일하게 나왔다. layer4에서 GAP 구조를 쓰는 경우 CAM과 Grad-CAM이 수학적으로 거의 동일한 결과를 내기 때문이라고 한다.

**2. easy가 hard보다 잘 나온다**

easy 평균(0.47)이 hard 평균(0.35)보다 높게 나왔다. 

**3. 분류 모델인데 위치도 어느 정도 찾는다**

ResNet-50은 "이게 무슨 개 종류냐"만 맞추도록 학습했는데, CAM을 뽑아보니 개가 있는 위치도 꽤 잘 찾아냈다.

---

### 트러블슈팅

##### `file_list`에 `.jpg`가 이미 포함되어 있어서 경로가 `.jpg.jpg`로 중복되는 문제가 있었다. 파일 존재 여부를 확인하는 방식으로 해결했다.
##### 클라우드 환경에서 `num_workers=2`로 설정했더니 shared memory 부족으로 Bus error가 났다. `num_workers=0`으로 바꿔서 해결했고 실험 결과에는 영향 없다.

---

### 느낀 점

##### epoch를 10으로 제한해서 모델 성능이 충분히 수렴하지 못했다. 30~50 epoch 정도 돌렸으면 IoU도 더 높게 나왔을 것 같다.
##### easy/hard 샘플을 임시 인덱스로 정해서 실제로 난이도 기준이 명확하지 않았다. 다음에는 이미지를 먼저 눈으로 보고 기준을 세워서 샘플을 뽑는 게 좋을 것 같다.
##### VGG팀 결과와 비교해보고 싶었는데 시간상 충분히 비교 분석을 못 한 게 아쉽다.

# 퍼실님 제가 그대로 올려보려고 했는데 아웃풋을 clear 안하면 자꾸 안올라가져서 일단 다 clear 하고 제출합니다!ㅠ